In [1]:
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
X_train = joblib.load("../data/X_train_raw.joblib")
X_test = joblib.load("../data/X_test_raw.joblib")

y_train = joblib.load("../data/y_train.joblib")
y_test = joblib.load("../data/y_test.joblib")

In [3]:
numerical_features = [
    "no_of_dependents",
    " income_annum",
    " loan_amount",
    " loan_term",
    " cibil_score",
    " residential_assets_value",
    " commercial_assets_value",
    " luxury_assets_value",
    " bank_asset_value",
    " total_assets",
    " loan_to_income_ratio",
    " asset_to_loan_ratio",
    " asset_to_income_ratio"
]

categorical_features = [
    " education",
    " self_employed",
]

In [4]:
y_train = y_train.map({
    "Rejected": 0,
    "Approved": 1
})


y_test = y_test.map({
    "Rejected": 0,
    "Approved": 1
})

In [5]:
y_train.isnull()

1224    False
478     False
3065    False
326     False
2991    False
        ...  
23      False
233     False
3298    False
1397    False
3222    False
Name:  loan_status, Length: 3415, dtype: bool

In [6]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

In [7]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [8]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [9]:
dt_model = DecisionTreeClassifier(
    random_state=42
)

In [10]:
dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", dt_model)
])

In [11]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [12]:
dt_cv_results = cross_validate(
    dt_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],
    return_train_score=True
)

In [13]:
print("Mean CV Accuracy :", dt_cv_results["test_accuracy"].mean())
print("Mean CV Precision:", dt_cv_results["test_precision"].mean())
print("Mean CV Recall   :", dt_cv_results["test_recall"].mean())
print("Mean CV F1       :", dt_cv_results["test_f1"].mean())
print("Mean CV ROC-AUC  :", dt_cv_results["test_roc_auc"].mean())

Mean CV Accuracy : 0.9982430453879942
Mean CV Precision: 0.9990610328638498
Mean CV Recall   : 0.9981176470588234
Mean CV F1       : 0.9985860109604854
Mean CV ROC-AUC  : 0.9982836297309621


In [14]:
print(X_train.columns.tolist())

['no_of_dependents', ' education', ' self_employed', ' income_annum', ' loan_amount', ' loan_term', ' cibil_score', ' residential_assets_value', ' commercial_assets_value', ' luxury_assets_value', ' bank_asset_value', ' total_assets', ' loan_to_income_ratio', ' asset_to_loan_ratio', ' asset_to_income_ratio']


In [15]:
print("Mean Train Accuracy :", dt_cv_results["train_accuracy"].mean())
print("Mean Train Precision:", dt_cv_results["train_precision"].mean())
print("Mean Train Recall   :", dt_cv_results["train_recall"].mean())
print("Mean Train F1       :", dt_cv_results["train_f1"].mean())
print("Mean Train ROC-AUC  :", dt_cv_results["train_roc_auc"].mean())

Mean Train Accuracy : 1.0
Mean Train Precision: 1.0
Mean Train Recall   : 1.0
Mean Train F1       : 1.0
Mean Train ROC-AUC  : 1.0


# Hyperparameter tuning max_depth

In [16]:
param_grid = {
    "model__max_depth": [
        2,
        3,
        4,
        5,
        6,
        8,
        10,
        12,
        15,
        20,
        None
    ]
}

In [17]:
dt_grid = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

In [18]:
dt_grid.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [2, 3, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,transformers,"[('num', ...), ('cat', ...)]"


In [19]:
print("Best max_depth:", dt_grid.best_params_)
print("Best CV ROC-AUC:", dt_grid.best_score_)

Best max_depth: {'model__max_depth': 3}
Best CV ROC-AUC: 0.9987906976744186


# tune min_samples_split

In [20]:
param_grid_split = {
    "model__max_depth": [3],
    "model__min_samples_split": [
        2,
        5,
        10,
        20,
        30,
        50
    ]
}

In [21]:
dt_grid_split = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=param_grid_split,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

In [22]:
dt_grid_split.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [3], 'model__min_samples_split': [2, 5, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,transformers,"[('num', ...), ('cat', ...)]"


In [23]:
print("Best parameters:", dt_grid_split.best_params_)
print("Best CV ROC-AUC:", dt_grid_split.best_score_)

Best parameters: {'model__max_depth': 3, 'model__min_samples_split': 2}
Best CV ROC-AUC: 0.9987906976744186


# tune min_samples_leaf

In [24]:
param_grid_leaf = {
    "model__max_depth": [3],
    "model__min_samples_split": [2],
    "model__min_samples_leaf": [
        1,
        2,
        5,
        10,
        20,
        30
    ]
}

In [25]:
dt_grid_leaf = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=param_grid_leaf,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

In [26]:
dt_grid_leaf.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [3], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,transformers,"[('num', ...), ('cat', ...)]"


In [27]:
print("Best parameters:", dt_grid_leaf.best_params_)
print("Best CV ROC-AUC:", dt_grid_leaf.best_score_)

Best parameters: {'model__max_depth': 3, 'model__min_samples_leaf': 10, 'model__min_samples_split': 2}
Best CV ROC-AUC: 0.9990031919744642


# tune max_features 

In [28]:
param_grid_features = {
    "model__max_depth": [3],
    "model__min_samples_split": [2],
    "model__min_samples_leaf": [10],
    "model__max_features": [
        None,
        "sqrt",
        "log2"
    ]
}

In [29]:
dt_grid_features = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=param_grid_features,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

In [30]:
dt_grid_features.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [3], 'model__max_features': [None, 'sqrt', ...], 'model__min_samples_leaf': [10], 'model__min_samples_split': [2]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,transformers,"[('num', ...), ('cat', ...)]"


In [31]:
print("Best parameters:", dt_grid_features.best_params_)
print("Best CV ROC-AUC:", dt_grid_features.best_score_)

Best parameters: {'model__max_depth': 3, 'model__max_features': None, 'model__min_samples_leaf': 10, 'model__min_samples_split': 2}
Best CV ROC-AUC: 0.9990031919744642


# Create the final Decision Tree

In [32]:
final_dt_model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=10,
    max_features=None,
    random_state=42
)

final_dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_dt_model)
])

final_dt_pipeline.fit(X_train, y_train)

y_pred_dt = final_dt_pipeline.predict(X_test)
y_prob_dt = final_dt_pipeline.predict_proba(X_test)[:, 1]

In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

dt_test_accuracy = accuracy_score(y_test, y_pred_dt)
dt_test_precision = precision_score(y_test, y_pred_dt)
dt_test_recall = recall_score(y_test, y_pred_dt)
dt_test_f1 = f1_score(y_test, y_pred_dt)
dt_test_roc_auc = roc_auc_score(y_test, y_prob_dt)

print("Decision Tree Test Accuracy :", dt_test_accuracy)
print("Decision Tree Test Precision:", dt_test_precision)
print("Decision Tree Test Recall   :", dt_test_recall)
print("Decision Tree Test F1       :", dt_test_f1)
print("Decision Tree Test ROC-AUC  :", dt_test_roc_auc)

Decision Tree Test Accuracy : 0.9988290398126464
Decision Tree Test Precision: 0.9981203007518797
Decision Tree Test Recall   : 1.0
Decision Tree Test F1       : 0.9990592662276576
Decision Tree Test ROC-AUC  : 0.9999912543072537


In [34]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_dt)

print(cm)

[[322   1]
 [  0 531]]


In [35]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_dt))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       323
           1       1.00      1.00      1.00       531

    accuracy                           1.00       854
   macro avg       1.00      1.00      1.00       854
weighted avg       1.00      1.00      1.00       854



In [37]:
print(dt_grid_features.best_estimator_)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['no_of_dependents',
                                                   ' income_annum',
                                                   ' loan_amount', ' loan_term',
                                                   ' cibil_score',
                                                   ' residential_assets_value',
                                                   ' commercial_assets_value',
                                                   ' luxury_assets_value',
                                                   ' bank_asset_value',
                                                   ' total_assets',
                                                   ' loan_to_income

In [38]:
import joblib

In [39]:
joblib.dump(
    dt_grid_features.best_estimator_,
    "../models/decision_tree_final_pipeline.pkl"
)

['../models/decision_tree_final_pipeline.pkl']

In [40]:
import os

print(
    os.path.exists(
        "../models/decision_tree_final_pipeline.pkl"
    )
)

True


In [41]:
file_path = "../models/decision_tree_final_pipeline.pkl"

print(os.path.getsize(file_path), "bytes")

6068 bytes
